# Concordances

In [1]:

import polars as pl
import polars.selectors as cs
from stack_data.utils import group_by_key_func

import polars_corpus as plc
import great_tables as gt

from amalgum import sentence_tags




In [ ]:
#bnc = pl.read_parquet("../bnc.parquet")
bnc = pl.scan_parquet("../bnc.parquet")
bnc.head(5).collect()

In [47]:
#m = plc.search(bnc, "little ( _AJ* | _NN* )* _NN*", pos_column="tag")
m = plc.search(bnc, "the _AJC the _AJC", pos_column='tag')

In [48]:
m

LazySearchResults<'the _AJC the _AJC'; 43 matches>

In [49]:
c = m.concordance("token", window=2)

In [51]:
c.with_columns(pl.col('token').list.join(" ").str.to_lowercase()).group_by("token").len().sort(by="len", descending=True)

token,len
str,u32
"""the higher the better""",4
"""the bigger the better""",4
"""the earlier the better""",4
"""the longer the better""",3
"""the quicker the better""",2
…,…
"""the faster the better""",1
"""the kinkier the better""",1
"""the messier the better""",1


In [52]:
m.view("token", chunk_column='sentence_tag')

<polars_corpus.view.ConcordanceWidget._create_widget.<locals>._ConcordanceWidget object at 0x109d6cc20>

In [16]:
owl = plc.search(bnc, "10 little owl")

In [21]:
owl.concordance(chunk_column='sentence_tag', metadata='file_id')

token_left_context,token,token_right_context,file_id
list[str],list[str],list[str],str
"[""Numbers"", ""and"", … "",""]","[""10"", ""little"", ""owl""]","["","", ""11"", … "".""]","""B2C"""
"[""Numbers"", ""and"", … "",""]","[""10"", ""little"", ""owl""]","["","", ""11"", … "".""]","""B2C"""
"[""Numbers"", ""and"", … "",""]","[""10"", ""little"", ""owl""]","["","", ""11"", … "".""]","""B2C"""
"[""Numbers"", ""and"", … "",""]","[""10"", ""little"", ""owl""]","["","", ""11"", … "".""]","""B2C"""
"[""Numbers"", ""and"", … "",""]","[""10"", ""little"", ""owl""]","["","", ""11"", … "".""]","""B2C"""
"[""Numbers"", ""and"", … "",""]","[""10"", ""little"", ""owl""]","["","", ""11"", … "".""]","""B2C"""


In [20]:
 o = bnc.filter(pl.col('token')=='owl').collect()
 o

token,lemma,pos,tag,sentence_tag,mode,text_type,file_id,speaker_id
str,str,str,str,str,str,str,str,str
"""owl""","""owl""","""SUBST""","""NN1""","""I""","""written""","""FICTION""","""A0L""",null
"""owl""","""owl""","""SUBST""","""NN1""","""I""","""written""","""FICTION""","""A0L""",null
"""owl""","""owl""","""SUBST""","""NN1""","""I""","""written""","""FICTION""","""A0L""",null
"""owl""","""owl""","""SUBST""","""NN1""","""I""","""written""","""FICTION""","""A0N""",null
"""owl""","""owl""","""SUBST""","""NN1""","""I""","""written""","""FICTION""","""A0U""",null
…,…,…,…,…,…,…,…,…
"""owl""","""owl""","""SUBST""","""NN1""","""I""","""spoken""","""CONVRSN""","""KD0""","""PS0HP"""
"""owl""","""owl""","""SUBST""","""NN1""","""I""","""spoken""","""CONVRSN""","""KD1""","""PS0JA"""
"""owl""","""owl""","""SUBST""","""NN1""","""I""","""spoken""","""CONVRSN""","""KDB""","""PS0L6"""


In [ ]:
m.concordance("token", chunk_column="sentence_tag")

In [ ]:
m.concordance("token").group_by("token").len()

In [ ]:
m.concordance("token", window=5)

In [ ]:
c.corpus.search("small _{SUBST}", pos_column='tag').concordance("token", window=5)

In [ ]:
c.corpus.search("( small | little ) _N*",pos_column='tag').concordance(
    "token", window=0).group_by('token').len().sort(by='len', descending=True)

In [ ]:
tbl = (
    verbs.concordance("token", window=20)
    .select(cs.all().list.join(" "))
    .with_columns(
        pl.col("token_left_context").str.tail(50),
        pl.col("token_right_context").str.head(50),
    )
    .style
)

tbl.tab_header(title=gt.md("*tweak* in the BNC")).cols_align(
    align="right", columns="token_left_context"
).cols_align(align="center", columns="token").cols_align(
    align="left", columns="token_right_context"
).tab_options(table_font_names=gt.system_fonts("industrial"))

In [ ]:
tbl = (
    verbs.concordance("token", chunk_column="sentence_tag")
    .select(cs.all().list.join(" "))
    .select(
        pl.concat_str(
            pl.col("token_left_context"),
            pl.lit(" **")
            + pl.col("token")
            + pl.lit("** ")
            + pl.col("token_right_context"),
        ).alias("sentence")
    )
    .style
)

tbl.tab_header(title=gt.md("*tweak* in the BNC")).cols_align(
    align="left", columns="sentence"
).fmt_markdown(columns="sentence")
#    .tab_options(table_font_names=gt.system_fonts("industrial")) \

In [ ]:
""